# Semantic Correspondence — Local pipeline

**Training and orchestration only.** All analysis (PCK tables, plots, qualitative figures, paper-ready exports) is done in [`AML_Analysis.ipynb`](AML_Analysis.ipynb).

Run this notebook from the repository root with `pip install -e ".[notebook]"` and a PyTorch wheel that matches your hardware.

**Resume:** re-run the cells; the pipeline auto-resumes via `runs/pipeline_state.json` and `checkpoints/*_resume.pt`.

## 1. Device probe (capability- and VRAM-driven)

In [ ]:
import os
import sys
from pathlib import Path

import torch

REPO_DIR = Path.cwd().resolve()
os.chdir(REPO_DIR)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Platform:", sys.platform)

VRAM_GB = 0.0
CUDA_MAJOR = 0
CUDA_MINOR = 0
GPU_NAME = ""

if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / (1024**3)
    CUDA_MAJOR, CUDA_MINOR = props.major, props.minor
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"CUDA: gpu_name={GPU_NAME!r}  VRAM_GB={VRAM_GB:.1f}  cap={CUDA_MAJOR}.{CUDA_MINOR}")
    BF16_CUDA = torch.cuda.is_bf16_supported() and CUDA_MAJOR >= 8
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    BF16_CUDA = False
    print("MPS detected: training will use fp32; unified memory imposes num_workers=1.")
else:
    DEVICE = "cpu"
    BF16_CUDA = False
    print("CPU only: training will be slow; consider reducing epochs in the config cell.")

if DEVICE == "cuda":
    if CUDA_MAJOR >= 8 and VRAM_GB >= 32:
        SPEED_TIER = "fast"
    elif CUDA_MAJOR >= 7 and VRAM_GB >= 12:
        SPEED_TIER = "medium"
    else:
        SPEED_TIER = "slow"
elif DEVICE == "mps":
    SPEED_TIER = "medium"
else:
    SPEED_TIER = "slow"

print(f"DEVICE={DEVICE}  SPEED_TIER={SPEED_TIER}")


## 2. Install package (idempotent)

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=str(REPO_DIR),
    check=True,
)
print("pip install OK")


## 3. Throughput tweaks (CUDA-only effect)

In [ ]:
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.allow_tf32 = True
    print("cuDNN benchmark + TF32 on")
try:
    torch.set_float32_matmul_precision("high")
except (RuntimeError, AttributeError):
    pass


## 4. SPair-71k dataset (download/extract if missing)

In [ ]:
import tarfile
import urllib.request

SPAIR_URL = "https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz"
DATA_DIR = REPO_DIR / "data"
SPAIR_ROOT = DATA_DIR / "SPair-71k"
TAR_PATH = DATA_DIR / "SPair-71k.tar.gz"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not SPAIR_ROOT.is_dir():
    if not TAR_PATH.is_file():
        print("Downloading:", SPAIR_URL)
        urllib.request.urlretrieve(SPAIR_URL, TAR_PATH)
    print("Extracting to:", DATA_DIR)
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        if sys.version_info >= (3, 12):
            tar.extractall(path=DATA_DIR, filter="data")
        else:
            tar.extractall(path=DATA_DIR)
else:
    print("Dataset present:", SPAIR_ROOT)

print("SPAIR_ROOT exists:", SPAIR_ROOT.is_dir())


## 5. Pretrained weights (DINOv2 / DINOv3 / SAM)

In [ ]:
subprocess.run([sys.executable, "scripts/download_pretrained_weights.py"], cwd=str(REPO_DIR), check=True)


## 6. Pipeline configuration

**Hyperparameters are FIXED** by the project specification (`documentation.md`, PDF):

| Hyperparameter | Value |
|---|---|
| Fine-tune batch size (DINO) | 20 |
| Fine-tune batch size (SAM) | 4 |
| LoRA batch size (DINO / SAM) | 20 / 4 |
| Fine-tune LR / weight decay | 5e-5 / 0.01 |
| LoRA LR / weight decay | 1e-3 / 0.0 |
| LoRA rank / alpha / target | 8 / 16.0 / MLP last 2 blocks |
| Last-blocks sweep | [1, 2, 4] |
| Epochs / patience | 50 / 7 |
| PCK alphas | 0.05, 0.1, 0.2 |
| WSA window / temperature | 5 / 1.0 |

**Hardware adaptation** is limited to runtime knobs (no hyperparameter changes):

| Knob | Selected from |
|---|---|
| `precision` | compute capability (Ampere+ → bf16; Volta/Turing → fp16; MPS/CPU → fp32) |
| `torch.compile` | Ampere+ with VRAM ≥ 12 GB |
| `num_workers` | MPS = 1 (unified memory); CUDA/CPU = `min(8, cpu_count // 2)` |
| `resume_save_interval` | `slow=50`, `medium=200`, `fast=500` batches |
| Batch sizes | PDF defaults on CUDA; **derogated** to smaller values on MPS / CPU only because PDF defaults OOM on unified-memory devices |

**Resume vs cold start:** the next cell sets `START_FROM_SCRATCH` automatically. If both `runs/` and `checkpoints/` exist under the repo root, the pipeline **resumes**. Delete either folder for a cold start.


In [ ]:
import yaml

RUNS_DIR = REPO_DIR / "runs"
CKPT_DIR = REPO_DIR / "checkpoints"
_has_resume_context = RUNS_DIR.is_dir() and CKPT_DIR.is_dir()
START_FROM_SCRATCH = not _has_resume_context

RUNTIME_DEVICE = DEVICE
if DEVICE == "cuda" and not BF16_CUDA:
    PRECISION = "fp16"
else:
    PRECISION = "auto"

if DEVICE == "mps":
    NUM_WORKERS = 1
else:
    NUM_WORKERS = max(2, min(8, (os.cpu_count() or 4) // 2))

COMPILE = (DEVICE == "cuda" and CUDA_MAJOR >= 8 and VRAM_GB >= 12.0)
RESUME_SAVE_INTERVAL = {"slow": 50, "medium": 200, "fast": 500}[SPEED_TIER]

FT_EPOCHS = 50
FT_PATIENCE = 7
LORA_EPOCHS = 50
LORA_PATIENCE = 7
FT_LR = 5e-5
FT_WEIGHT_DECAY = 0.01
LORA_LR = 1e-3
LORA_ALPHA = 16.0
LORA_RANK = 8
LORA_LAST_BLOCKS = 2
LAST_BLOCKS_LIST = [1, 2, 4]
IMAGE_SIZE_BY_BACKBONE = {
    "dinov2_vitb14": [518, 518],
    "dinov3_vitb16": [512, 512],
    "sam_vit_b": [512, 512],
}
LOG_BATCH_INTERVAL = 50
RUN_PYTEST = False

if DEVICE == "cuda":
    FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 20, "dinov3_vitb16": 20, "sam_vit_b": 4}
    LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 20, "dinov3_vitb16": 20, "sam_vit_b": 4}
    FT_FALLBACK = 20
    LORA_FALLBACK = 20
elif DEVICE == "mps":
    FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 8, "dinov3_vitb16": 8, "sam_vit_b": 3}
    LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 12, "dinov3_vitb16": 12, "sam_vit_b": 3}
    FT_FALLBACK = 8
    LORA_FALLBACK = 12
else:
    FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 4, "dinov3_vitb16": 4, "sam_vit_b": 2}
    LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 4, "dinov3_vitb16": 4, "sam_vit_b": 2}
    FT_FALLBACK = 4
    LORA_FALLBACK = 4

cfg = {
    "paths": {
        "spair_root": str(REPO_DIR / "data" / "SPair-71k"),
        "checkpoint_dir": str(CKPT_DIR),
    },
    "runtime": {
        "device": RUNTIME_DEVICE,
        "precision": PRECISION,
        "compile": COMPILE,
        "num_workers": NUM_WORKERS,
        "preprocess": "FIXED_RESIZE",
        "image_size_by_backbone": IMAGE_SIZE_BY_BACKBONE,
        "limit_pairs": 0,
        "eval_split": "test",
        "alphas": [0.05, 0.1, 0.2],
        "wsa_window": 5,
        "wsa_temperature": 1.0,
        "log_batch_interval": LOG_BATCH_INTERVAL,
        "resume_save_interval": RESUME_SAVE_INTERVAL,
    },
    "finetune": {
        "last_blocks": LAST_BLOCKS_LIST,
        "epochs": FT_EPOCHS,
        "patience": FT_PATIENCE,
        "batch_size": FT_FALLBACK,
        "batch_size_by_backbone": FT_BATCH_SIZE_BY_BACKBONE,
        "lr": FT_LR,
        "weight_decay": FT_WEIGHT_DECAY,
        "dinov2_weights": str(CKPT_DIR / "dinov2_vitb14_pretrain.pth"),
        "dinov3_weights": str(CKPT_DIR / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"),
        "sam_checkpoint": str(CKPT_DIR / "sam_vit_b_01ec64.pth"),
    },
    "lora": {
        "rank": LORA_RANK,
        "alpha": LORA_ALPHA,
        "lr": LORA_LR,
        "last_blocks": LORA_LAST_BLOCKS,
        "epochs": LORA_EPOCHS,
        "patience": LORA_PATIENCE,
        "batch_size": LORA_FALLBACK,
        "batch_size_by_backbone": LORA_BATCH_SIZE_BY_BACKBONE,
    },
    "workflow_toggles": {
        "run_verify_dataset": True,
        "train_finetune": [True, True, True],
        "train_lora": [True, True, True],
        "run_eval_baseline": [True, True, True],
        "run_eval_baseline_wsa": [True, True, True],
        "run_eval_finetuned_checkpoint": [True, True, True],
        "run_eval_lora_checkpoint": [True, True, True],
        "run_eval_finetuned_wsa": [True, True, True],
        "run_eval_lora_wsa": [True, True, True],
        "run_export_metrics_tables": True,
        "run_pytest": RUN_PYTEST,
        "pipeline_resume": not START_FROM_SCRATCH,
    },
}

cfg_path = REPO_DIR / "config.yaml"
with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)

print("Written:", cfg_path)
print(f"device={RUNTIME_DEVICE}  speed_tier={SPEED_TIER}  precision={PRECISION}  "
      f"compile={COMPILE}  num_workers={NUM_WORKERS}  resume_save_interval={RESUME_SAVE_INTERVAL}")
print("Pipeline:", "resume" if _has_resume_context else "cold start",
      "| START_FROM_SCRATCH =", START_FROM_SCRATCH)
print("FT batches:", FT_BATCH_SIZE_BY_BACKBONE, "LoRA:", LORA_BATCH_SIZE_BY_BACKBONE)


## 7. Run pipeline (live dashboard)

In [ ]:
import json
import threading
from collections import deque

from IPython.display import clear_output

os.chdir(REPO_DIR)

env = dict(os.environ)
if START_FROM_SCRATCH:
    env["SEMANTIC_CORRESPONDENCE_PIPELINE_RESET"] = "1"
else:
    env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_RESET", None)
env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_LOG_FILE_ONLY", None)
env["PYTHONUNBUFFERED"] = "1"
if DEVICE == "mps":
    # Some MPS ops (e.g. grid_sampler_2d_backward) lack kernels; allow CPU fallback.
    env["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
_pp = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = f"{REPO_DIR}{os.pathsep}{_pp}" if _pp else str(REPO_DIR)

STAGE_EVENTS_PATH = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
TAIL_LINES = 40
REFRESH_SECONDS = 3.0
cmd = [sys.executable, "-u", "scripts/run_pipeline.py", "--config", "config.yaml"]

_output_lines: deque = deque(maxlen=TAIL_LINES)
_proc_done = threading.Event()
_return_code: list = []


def _stream_subprocess() -> None:
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env, cwd=str(REPO_DIR),
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        _output_lines.append(line.rstrip())
    _return_code.append(proc.wait())
    _proc_done.set()


threading.Thread(target=_stream_subprocess, daemon=True).start()


def _read_stage_events() -> list:
    if not STAGE_EVENTS_PATH.is_file():
        return []
    events = []
    with open(STAGE_EVENTS_PATH, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            try:
                events.append(json.loads(raw))
            except json.JSONDecodeError:
                pass
    return events


def _render(events: list) -> None:
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    latest_start = next((e for e in reversed(events) if e.get("action") == "start"), None)
    current = latest_start["stage_id"] if latest_start else "(waiting...)"
    rc = _return_code[0] if _return_code else None
    if not _proc_done.is_set():
        status = "RUNNING"
    elif rc == 0:
        status = "COMPLETED"
    else:
        status = f"FAILED (exit={rc})"
    sep = "=" * 64
    print(sep)
    print(f"  Pipeline: {status}")
    print(f"  Current stage: {current}")
    print(f"  Completed: {len(done):3d}  Skipped: {len(skipped):3d}  Failed: {len(failed):3d}")
    if done:
        tail = done[-6:]
        suffix = "..." if len(done) > 6 else ""
        print(f"  Last done: {', '.join(tail)}{suffix}")
    if failed:
        print(f"  FAILED: {', '.join(failed)}")
    print(sep)
    print(f"--- last {TAIL_LINES} log lines ---")
    for line in _output_lines:
        print(line)


while not _proc_done.is_set():
    clear_output(wait=True)
    _render(_read_stage_events())
    _proc_done.wait(timeout=REFRESH_SECONDS)

clear_output(wait=True)
_render(_read_stage_events())

rc = _return_code[0] if _return_code else -1
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)
print("\nPipeline completed successfully.")


## 8. Stage summary (sanity check)

In [ ]:
import pandas as pd

p = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
if not p.is_file():
    print("No file:", p)
else:
    events = []
    with open(p, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if raw:
                try:
                    events.append(json.loads(raw))
                except json.JSONDecodeError:
                    pass
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    print(f"events={len(events)} done={len(done)} skipped={len(skipped)} failed={len(failed)}")
    if failed:
        print("Failed:", failed)
    display(pd.DataFrame(events).tail(20))


---

**Pipeline completed.** Open [`AML_Analysis.ipynb`](AML_Analysis.ipynb) to produce paper-ready tables, plots, and qualitative figures from `runs/pipeline_exports/` and `checkpoints/`.